# Offline d1/d2/d3 validation harness

Scores any retrieval method on synthetic proxies of all three difficulty levels,
built from dataset1's labelled pairs. **Run All** and read the table at the bottom.

- Lives in `amine/` alongside `eval_harness.py` (kernel cwd = this folder).
- Cell 2 symlinks `data/` and `out/` here so the data + outputs show in the left panel.
- Edit `EMBEDDERS` in `eval_harness.py` to plug in new methods, then re-run.
- Last cell gives download links for the outputs.

In [1]:
import subprocess, sys
# nibabel+scipy for the reference embedders; torch+monai for the learned/Swin embedder.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel>=5.3', 'scipy', 'monai>=1.3'], check=True)
print('deps ready')

deps ready


In [2]:
import os
# Make the input data and output dir browsable in the file panel on the left:
# create symlinks INSIDE this folder (amine/) pointing at the real locations.
# One-time per container; harmless if they already exist.
for name, target in [('data', '/workspace/data/ehl'), ('out', '/workspace/out')]:
    if os.path.islink(name) or os.path.exists(name):
        print('exists:', name, '->', os.path.realpath(name))
        continue
    if os.path.isdir(target):
        os.symlink(target, name)
        print('linked:', name, '->', target)
    else:
        print('SKIP', name, '- target not found:', target, '(fix the path if your mount differs)')

exists: data -> /workspace/data/ehl
exists: out -> /workspace/out


In [3]:
import os
os.environ['DATA_ROOT'] = '/workspace/data/ehl'
os.environ.setdefault('N_VAL', '60')   # held-out dataset1 pairs to score on
os.environ.setdefault('GRID', '96')    # resample cube size

# --- LEARNED MODEL: MIND-input contrastive embedder (the d2/d3 model) ---------
# Feeds modality-invariant MIND fields to a 3D CNN trained CLIP-style with heavy
# rigid+elastic+resection augmentation -> a deformation-robust global embedding.
os.environ['SKIP_LEARNED'] = '1'         # skip the old raw-intensity Swin/CNN embedder
os.environ['USE_MIND_LEARNED'] = '1'     # train + score mind_embedder (the new model)
# knobs (defaults are sensible; bump epochs if loss is still falling):
# os.environ['MIND_EPOCHS']='200'; os.environ['MIND_RESECT']='0.5'; os.environ['MIND_TTA']='8'

# also score the training-free GPU rankers for reference (nmi/mind/dense_mind):
assert os.path.isdir(os.environ['DATA_ROOT']), 'data root not found - fix DATA_ROOT above'
for f in ('eval_harness.py','rankers.py','mind_embedder.py','learned_embedder.py'):
    assert os.path.exists(f), f're-upload {f} here'
print('ready: training mind_learned + scoring rankers on', os.environ['DATA_ROOT'])

ready: training mind_learned + scoring rankers on /workspace/data/ehl


In [4]:
import importlib, eval_harness
importlib.reload(eval_harness)   # pick up edits without restarting the kernel
results = eval_harness.evaluate()

DATA_ROOT=/workspace/data/ehl  indexed=1454  val_pairs=60  train_pairs=290  grid=96
[mind] device=cuda pairs=290 cfg={'epochs': 200, 'batch': 64, 'dim': 128, 'width': 24, 'lr': 0.0003, 'rot': 20.0, 'elastic': 0.08, 'resect': 0.5, 'wd': 0.01, 'dil': 2, 'tta': 8}
[mind] epoch 001 loss=4.0805 (10s)
[mind] epoch 025 loss=4.0847 (37s)
[mind] epoch 050 loss=4.0678 (66s)
[mind] epoch 075 loss=4.0394 (94s)
[mind] epoch 100 loss=4.0356 (123s)
[mind] epoch 125 loss=3.9977 (151s)
[mind] epoch 150 loss=3.9479 (180s)
[mind] epoch 175 loss=3.8047 (208s)
[mind] epoch 200 loss=3.7135 (237s)
  loaded 10/60 pairs  (0s)
  loaded 20/60 pairs  (0s)
  loaded 30/60 pairs  (0s)
  loaded 40/60 pairs  (0s)
  loaded 50/60 pairs  (0s)
  loaded 60/60 pairs  (0s)
[rankers] ['nmi', 'gradcos', 'nmi_grad', 'mind', 'dense_mind'] on device=cuda
  [d1] intensity    MRR=0.350
  [d1] edges        MRR=0.087
  [d1] fingerprint  MRR=0.186
  [d1] mind_learned MRR=0.170
  [d1] nmi          MRR=0.522
  [d1] gradcos      MRR=0.50

In [5]:
import os, shutil
from IPython.display import FileLink, display
# Click these links to download outputs to your computer.
# (submission.csv is produced by run_baseline.ipynb; searched in a few common spots.)
def offer(fname, candidates):
    for c in candidates:
        if os.path.exists(c):
            if os.path.abspath(c) != os.path.abspath(fname):
                shutil.copy(c, fname)   # bring it next to this notebook so the link resolves
            display(FileLink(fname))
            return
    print('not found yet:', fname)

offer('eval_results.md', ['eval_results.md'])
offer('submission.csv', ['submission.csv', 'out/submission.csv',
                         '/workspace/out/submission.csv', '/root/submission.csv'])

/shared-docker/amine/eval_results.md

/shared-docker/amine/submission.csv

In [6]:
from IPython.display import Markdown
Markdown(open('eval_results.md').read())

# Offline validation results

- DATA_ROOT: `/workspace/data/ehl`
- held-out dataset1 pairs: **60**, grid 96³, seed 20260627
- d1 = modality gap only · d2 = + deformation · d3 = + resection/bias/gamma
- macro = mean(d1,d2,d3). Higher is better (random ≈ 1/gallery-ish).

| embedder | d1 | d2 | d3 | **macro** |
|---|---|---|---|---|
| mind | 0.688 | 0.169 | 0.155 | **0.338** |
| dense_mind | 0.703 | 0.146 | 0.163 | **0.338** |
| nmi_grad | 0.567 | 0.149 | 0.146 | **0.287** |
| nmi | 0.522 | 0.149 | 0.129 | **0.267** |
| gradcos | 0.508 | 0.133 | 0.129 | **0.257** |
| intensity | 0.350 | 0.145 | 0.094 | **0.196** |
| mind_learned | 0.170 | 0.161 | 0.138 | **0.157** |
| fingerprint | 0.186 | 0.136 | 0.135 | **0.152** |
| edges | 0.087 | 0.076 | 0.079 | **0.081** |

_generated 2026-06-27 23:37:16_